# DEA - Nordrhein-Westfalen

In [ ]:
import patoolib
from tqdm import tqdm
import pandas as pd
import numpy as np
import pyproj
from camelsp import get_metadata

from camels_de1h import get_input_path, get_nuts_id_from_provider_id, Station1h

## Extract data

In [2]:
input_path = get_input_path("DEA")

data_path_q = input_path / "Abfluss.7z"
data_path_w = input_path / "Wasserstand.7z"

if not (input_path / "Abfluss_csv").exists():
    patoolib.extract_archive(str(data_path_q), outdir=str(input_path), interactive=False, verbosity=-1)
    print("Discharge data extracted")
else:
    print("Discharge data already extracted")

if not (input_path / "Wasserstand").exists():
    patoolib.extract_archive(str(data_path_w), outdir=str(input_path), interactive=False, verbosity=-1)
    print("Water level data extracted")
else:
    print("Water level data already extracted")

Discharge data already extracted
Water level data already extracted


In [14]:
# Export_Open_Data_Abfluss_ 4761500000100_20241004_002355.csv
ids_q = [f.stem.split("_")[4] for f in (input_path / "Abfluss_csv").glob("Export_Open_Data_Abfluss_*.csv")]
ids_w = [f.stem.split("_")[4] for f in (input_path / "Wasserstand").glob("Export_Open_Data_Wasserstand_*.csv")]

ids = sorted(list(set(ids_q).intersection(set(ids_w))))

print(f"Number of stations with only discharge data: {len(set(ids_q) - set(ids))}")
print(f"Number of stations with only water level data: {len(set(ids_w) - set(ids))}")
print(f"Number of stations with both discharge and water level data: {len(ids)}")

Number of stations with only discharge data: 1
Number of stations with only water level data: 1
Number of stations with both discharge and water level data: 239


## Parse data

> 1. Die Höhenangabe in Zeile 18 der Zeitreihen kannst du ignorieren. Als Höhenangabe relevant ist die Angabe des Pegelnullpunkts in Zeile 6.
> 2. Bei der Höhenangabe des Pegelnullpunkts gibt es leider zwei unterschiedliche Referenzsysteme (DHHN92 und DHHN2016). Die müsstest du bei den entsprechenden Pegeln also umrechnen. 
> 3. Die meisten Pegel haben als W- und Q-Einheit cm und m3/s, manche aber auch mm und l/s, weshalb du auch hier eine Umrechnung vornehmen müsstest. 

In [5]:
def parse_station_data(id: str, aggregate_hourly: bool) -> [pd.DataFrame, pd.DataFrame]:
    # get the discharge and water level file names
    fname_q = input_path / "Abfluss_csv" / f"Export_Open_Data_Abfluss_{id}_20241004_002355.csv" if id != "4761500000100" else input_path / "Abfluss_csv" / f"Export_Open_Data_Abfluss_ {id}_20241004_002355.csv" # there is a space before the id in the file name
    fname_w = input_path / "Wasserstand" / f"Export_Open_Data_Wasserstand_{id}_20241004_062725.csv"

    # read the header of the discharge file
    header_q = pd.read_csv(fname_q, sep=";", encoding="latin1", nrows=32, header=None)
    header_w = pd.read_csv(fname_w, sep=";", encoding="latin1", nrows=32, header=None)

    # merge the metadata headers, if the values are the same, keep only one, otherwise keep both and add Q or W to the column name
    dict_q = dict(zip(header_q[0], header_q[1]))
    dict_w = dict(zip(header_w[0], header_w[1]))

    # Create merged dictionary
    merged = {}

    # Compare and merge
    for key in dict_q.keys():
        if key in dict_w:
            if dict_q[key] == dict_w[key]:
                merged[key] = dict_q[key]
            else:
                merged[f"{key}_Q"] = dict_q[key]
                merged[f"{key}_W"] = dict_w[key]
        else:
            merged[f"{key}_Q"] = dict_q[key]

    # Add any remaining keys from header_w
    for key in dict_w.keys():
        if key not in dict_q:
            merged[f"{key}_W"] = dict_w[key]

    # Create final DataFrame
    meta = pd.DataFrame([merged])

    # read the data
    data_q = pd.read_csv(fname_q, sep=";", encoding="latin1", skiprows=32, header=None, decimal=",", na_values=["LUECKE"],
                        parse_dates=[0], date_format="%d.%m.%Y %H:%M:%S")
    data_q.columns = ["date", "discharge_vol_obs"]

    # datetime format is "%d.%m.%Y %H:%M" for some stations
    if isinstance(data_q["date"][0], str):
        data_q["date"] = pd.to_datetime(data_q["date"], format="%d.%m.%Y %H:%M")

    data_w = pd.read_csv(fname_w, sep=";", encoding="latin1", skiprows=32, header=None, decimal=",", na_values=["LUECKE"],
                         parse_dates=[0], date_format="%d.%m.%Y %H:%M:%S")
    data_w.columns = ["date", "water_level_obs"]

    # check units
    if meta["Einheit_Q"].values[0] == "l/s":
        data_q["discharge_vol_obs"] = data_q["discharge_vol_obs"] / 1000

    if meta["Einheit_W"].values[0] == "mm":
        data_w["water_level_obs"] = data_w["water_level_obs"] / 10

    data = pd.merge(data_q, data_w, on="date", how="outer").sort_values("date")

    # drop dates where both discharge and water level are missing
    data = data.dropna(subset=["discharge_vol_obs", "water_level_obs"], how="all").reset_index(drop=True)

    if aggregate_hourly:
        data = data.set_index("date").resample("h", label="right", closed="right").mean().reset_index()

    # transform date time to UTC+0 (data is in UTC+1)
    data["date"] = data["date"] - pd.Timedelta(hours=1)
    data["date"] = data["date"].dt.tz_localize("UTC")

    return meta, data

parse_station_data("4761500000100", aggregate_hourly=True)

(   Station Stationsnummer  Unterbezeichnung_Q  Unterbezeichnung_W  Gewässer_Q  \
 0  Fiestel  4761500000100                 NaN                 NaN         NaN   
 
    Gewässer_W Einzugsgebiet            Pegelnullpunkt Parameter_Q  \
 0         NaN    102,24 km²  43,889 mDHHN92 (aktuell)     Abfluss   
 
    Parameter_W  ... REIHENART VERSION       X XDISTANZ XEINHEIT_Q XEINHEIT_W  \
 0  Wasserstand  ...         Z       0  469941        E        NaN        NaN   
 
   XFAKTOR        Y YTYP_Q  YTYP_W  
 0     180  5800420    NaN     NaN  
 
 [1 rows x 46 columns],
                             date  discharge_vol_obs  water_level_obs
 0      1968-10-31 23:00:00+00:00              0.889         80.99300
 1      1968-11-01 00:00:00+00:00              0.889         81.00000
 2      1968-11-01 01:00:00+00:00              0.889         81.00000
 3      1968-11-01 02:00:00+00:00              0.889         81.00000
 4      1968-11-01 03:00:00+00:00              0.889         81.00000
 ...    

In [6]:
def check_metadata_consistency(meta, camelsp_meta, raw_meta, id: str) -> dict:
    """
    Check consistency of metadata fields across different metadata sources.
    Waterbody field is only compared between raw and camelsp.
    
    """
    inconsistencies = {}
    
    fields = {
        "provider_id": {
            "meta": "Stationsnummer",
            "camelsp": "provider_id",
            "raw": "station_id"
        },
        "gauge_name": {
            "meta": "Station",
            "camelsp": "gauge_name",
            "raw": "station_name"
        },
        "waterbody": {
            "camelsp": "waterbody_name",
            "raw": "Gewässer"
        },
        "area": {
            "meta": "Einzugsgebiet",
            "camelsp": "area",
            "raw": "EZG_Groeße"
        }
    }

    if camelsp_meta.empty:
        # Two-way comparisons (skip waterbody)
        for field_key, field_info in fields.items():
            if field_key == "waterbody":
                continue
                
            meta_val = meta[field_info["meta"]].values[0]
            raw_val = raw_meta[field_info["raw"]].values[0]
                
            if meta_val != raw_val:
                if id in inconsistencies:
                    inconsistencies[id].update({field_key: {
                        "meta": meta_val,
                        "raw": raw_val
                    }})
                else:
                    inconsistencies[id] = {field_key: {
                        "meta": meta_val,
                        "raw": raw_val
                    }}
    else:
        # Three-way comparisons
        for field_key, field_info in fields.items():
            if field_key == "waterbody":
                # Compare only raw and camelsp for waterbody
                raw_val = raw_meta[field_info["raw"]].values[0]
                camelsp_val = camelsp_meta[field_info["camelsp"]].values[0]
                
                if raw_val != camelsp_val:
                    if id in inconsistencies:
                        inconsistencies[id].update({field_key: {
                            "raw": raw_val,
                            "camelsp": camelsp_val
                        }})
                    else:
                        inconsistencies[id] = {field_key: {
                            "raw": raw_val,
                            "camelsp": camelsp_val
                        }}
                continue
            
            # Regular three-way comparison for other fields
            meta_val = meta[field_info["meta"]].values[0]
            camelsp_val = camelsp_meta[field_info["camelsp"]].values[0]
            raw_val = raw_meta[field_info["raw"]].values[0]
            
            if not (meta_val == camelsp_val == raw_val):
                if id in inconsistencies:
                    inconsistencies[id].update({field_key: {
                        "meta": meta_val,
                        "camelsp": camelsp_val,
                        "raw": raw_val
                    }})
                else:
                    inconsistencies[id] = {field_key: {
                        "meta": meta_val,
                        "camelsp": camelsp_val,
                        "raw": raw_val
                    }}
    
    return inconsistencies

In [ ]:
# get the metadata from CAMELS-DE daily
camelsp_meta_all = get_metadata()

# for NRW there is a raw metadata file that we can use if the station is not in the CAMELS-DE metadata
raw_meta_all = pd.read_excel(input_path / "Hydrologische-Stationen-NRW_EPSG25832_mitGewaessernamen.xlsx")

raw_meta_all["station_id"] = raw_meta_all["station_id"].astype(str)

metadata_inconsistencies = {}

not_in_camelsp = []

for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    try:
        gauge_id = get_nuts_id_from_provider_id(id, "DEA", add_missing=False)
    except ValueError:
        not_in_camelsp.append(id)
        continue

    # parse metadata and data for the station
    meta, data = parse_station_data(id, aggregate_hourly=False)

    meta["Pegelnullpunkt"] = float(meta["Pegelnullpunkt"].values[0].split(" ")[0].replace(",", "."))
    meta["Einzugsgebiet"] = float(meta["Einzugsgebiet"].values[0].split(" ")[0].replace(",", "."))

    # get the metadata of the station
    camelsp_meta = camelsp_meta_all[camelsp_meta_all["camels_id"] == gauge_id]
    raw_meta = raw_meta_all[raw_meta_all["station_id"] == id]

    # check the consistency of the metadata
    try:
        inconsistencies = check_metadata_consistency(meta, camelsp_meta, raw_meta, id)
    except IndexError:
        print(f"Error for station {id}")
        continue
    if inconsistencies:
        metadata_inconsistencies.update(inconsistencies)

metadata_inconsistencies

# make it a panads DataFrame with the station_id as index
pd.DataFrame(metadata_inconsistencies).T

Most of the discrepancies in river names come from encoding issues which I cannot fix, but wen can just use the camelsp meta here without the encoding issues.

But there are also some potential errors in camelsp meta, which I manually checked:
- `2721230000100`	{'raw': 'Ochsenbach', 'camelsp': 'Werthenbach'} -> is actually `Ochsenbach`
- `2721459000100`	{'raw': 'Ferndorfbach', 'camelsp': 'Ferndorf'} -> `Ferndorf` is correct / fine
- `2721490000100`	{'raw': 'Ferndorfbach', 'camelsp': 'Ferndorf'} -> `Ferndorf` is correct / fine
- `2762510000300`   {'raw': 'Bormelsbach', 'camelsp': 'Möhne'} -> is actually `Bormelsbach`
- `2766424000100`	{'raw': 'Rehsiepen', 'camelsp': 'Killmecke'} -> is actually `Rehsiepen`
- `2783250000100`	{'raw': 'Thune', 'camelsp': 'Strothe'} -> is actually `Thune`
- `2799222000100`   {'raw': 'Groesbecker-Bach', 'camelsp': 'Groesbeeker Bach'} -> is actually `Groesbecker-Bach`
- `2823100000100`	{'raw': 'Weidenbach', 'camelsp': 'Rur'} -> is actually `Weidenbach`
- `3128100000100`	{'raw': 'Wapelbach', 'camelsp': 'Wapel'} -> `Wapel` is correct / fine
- `4566900000100`	{'raw': 'Diestelbach', 'camelsp': 'Diestel'} -> `Diestel` is correct / fine

Where adjustments are necessary, we will use the raw meta information.  
Discrepancies in area are that small that we can ignore them.

## Parse data and metadata for the stations and save to ouput directory

We priotiize the CAMELS-DE (camelsp) metadata information over the newly provided metadata information to be in line with the CAMELS-DE catchment area information.  
An exception is the river name, where we use the raw meta information if the camelsp meta information is incorrect (see above).

In [18]:
ids_river_names_from_raw = ["2721230000100", "2721459000100", "2721490000100", "2762510000300", "2766424000100", "2783250000100", "2799222000100", "2823100000100", "3128100000100", "4566900000100"]

# Define lookup table for missing river names
river_name_mapping = {
    "2761490000100": "Henne",
    "2763830000100": "Rammbach",
    "3179000000100": "Ems"
}

In [ ]:
ids_q = sorted([str(p).split("_")[-3].strip() for p in (input_path / "Abfluss_csv").glob("*.csv")])
ids_w = sorted([str(p).split("_")[-3].strip() for p in (input_path / "Wasserstand").glob("*.csv")])

if ids_q == ids_w:
    ids = ids_q
    
# get the metadata from CAMELS-DE daily
camelsp_meta_all = get_metadata()

# for Sachsen-Anhalt there is a raw metadata file that we can use if the station is not in the CAMELS-DE metadata
raw_meta_all = pd.read_excel(input_path / "Hydrologische-Stationen-NRW_EPSG25832_mitGewaessernamen.xlsx")
raw_meta_all["station_id"] = raw_meta_all["station_id"].astype(str)

for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    gauge_id = get_nuts_id_from_provider_id(id, "DEA", add_missing=True)

    # parse metadata and data for the station
    meta, data = parse_station_data(id, aggregate_hourly=True)

    # get the metadata of the station
    camelsp_meta = camelsp_meta_all[camelsp_meta_all["camels_id"] == gauge_id]
    raw_meta = raw_meta_all[raw_meta_all["station_id"] == id]

    if not camelsp_meta.empty:
        # handle the IDs where the waterbody name is not correct in the CAMELS-DE metadata
        if id in ids_river_names_from_raw:
            waterbody_name = raw_meta["Gewässer"].values[0]
        # check if river name is in the mapping
        elif id in river_name_mapping:
            waterbody_name = river_name_mapping[id]
        else:
            waterbody_name = camelsp_meta["waterbody_name"].values[0]

        # set metadata values from camelsp metadata
        station_meta = {
            "gauge_id": camelsp_meta["camels_id"].values[0],
            "provider_id": camelsp_meta["provider_id"].values[0],
            "gauge_name": camelsp_meta["gauge_name"].values[0],
            "waterbody_name": waterbody_name,
            "federal_state": camelsp_meta["federal_state"].values[0],
            "gauge_lon": camelsp_meta["lon"].values[0],
            "gauge_lat": camelsp_meta["lat"].values[0],
            "gauge_easting": camelsp_meta["x"].values[0],
            "gauge_northing": camelsp_meta["y"].values[0],
            "gauge_elev_metadata": camelsp_meta["gauge_elevation"].values[0],
            "area_metadata": camelsp_meta["area"].values[0],
            "part_of_camelsp": True,
        }
    elif not raw_meta.empty:
        # parse the location
        easting, northing = raw_meta["KOORDX"].values[0], raw_meta["KOORDYY"].values[0]

        # from EPSG:25832 to EPSG:4326
        transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:4326", always_xy=True)
        lon, lat = transformer.transform(easting, northing)

        # from EPSG:31466 to EPSG:3035
        transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:3035", always_xy=True)
        easting, northing = transformer.transform(easting, northing)

        # get waterbody name from mapping or raw metadata
        if id in river_name_mapping:
            waterbody_name = river_name_mapping[id]
        else:
            waterbody_name = raw_meta["Gewässer"].values[0]

        # if the station is not in the camelsp metadata, use the raw metadata
        station_meta = {
            "gauge_id": gauge_id,
            "provider_id": id,
            "gauge_name": raw_meta["station_name"].values[0],
            "waterbody_name": waterbody_name,
            "federal_state": "Nordrhein-Westfalen",
            "gauge_lon": lon,
            "gauge_lat": lat,
            "gauge_easting": easting,
            "gauge_northing": northing,
            "gauge_elev_metadata": raw_meta["Nullpunkt"].values[0],
            "area_metadata": raw_meta["EZG_Groeße"].values[0],
            "part_of_camelsp": False,
        }
    else:
        # set metadata values from the metadata that came with the data
        # parse the waterbody name if available
        if id in river_name_mapping:
            waterbody_name = river_name_mapping[id]
        elif id in ids_river_names_from_raw:
            waterbody_name = meta["Gewässer"].values[0]
        elif "Gewässer" in meta.columns:
            waterbody_name = meta["Gewässer"].values[0]
        elif "Gewässer_Q" in meta.columns and not np.isnan(meta["Gewässer_Q"].values[0]):
            waterbody_name = meta["Gewässer_Q"].values[0]
        elif "Gewässer_W" in meta.columns and not np.isnan(meta["Gewässer_W"].values[0]):
            waterbody_name = meta["Gewässer_W"].values[0]
        else:
            waterbody_name = None

        # parse the location
        easting, northing = meta["X"].values[0], meta["Y"].values[0]

        # from EPSG:25832 to EPSG:4326
        transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:4326", always_xy=True)
        lon, lat = transformer.transform(easting, northing)

        # from EPSG:31466 to EPSG:3035
        transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:3035", always_xy=True)
        easting, northing = transformer.transform(easting, northing)

        station_meta = {
            "gauge_id": gauge_id,
            "provider_id": id,
            "gauge_name": meta["Station"].values[0],
            "waterbody_name": waterbody_name,
            "federal_state": "Nordrhein-Westfalen",
            "gauge_lon": lon,
            "gauge_lat": lat,
            "gauge_easting": easting,
            "gauge_northing": northing,
            "gauge_elev_metadata": float(meta["Pegelnullpunkt"].values[0].split(" ")[0].replace(",", ".")),
            "area_metadata": float(meta["Einzugsgebiet"].values[0].split(" ")[0].replace(",", ".")),
            "part_of_camelsp": False,
        } 
    
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(pd.concat([meta.reset_index(drop=True), raw_meta.reset_index(drop=True)], axis=1))